# Chapter 6 — Implementing MCP Client (Connecting Agent to MCP Server)

Now you’ll learn the most important integration step:

> How your agent connects to an MCP server and uses tools dynamically.

You already built the MCP server in Chapter 5.

Now we connect to it from an agent.

---

# 1️⃣ What MCP Client Does

MCP Client acts as bridge between:

```text
Agent (LLM + reasoning)
        ↓
MCP Client
        ↓
MCP Server (tools/resources/prompts)
```

Client responsibilities:

* Connect to server
* Discover tools
* Send tool calls
* Receive results
* Return results to agent

---

# 2️⃣ Internal Flow (Very Important)

Full lifecycle:

```text
Step 1: Client connects to MCP server

Step 2: Client asks → tools/list

Step 3: Server returns tool schemas

Step 4: Client gives tool schemas to LLM

Step 5: LLM selects tool

Step 6: Client sends tools/call

Step 7: Server executes tool

Step 8: Result returned to agent
```

This integrates directly into ReAct cycle.

---

# 3️⃣ Installing MCP Client SDK

Same SDK provides client:

```bash
pip install mcp
```

---

# 4️⃣ Connecting to MCP Server (Basic Example)

Assume your server is running locally.

### Client Code:

```python
import asyncio
from mcp.client.session import ClientSession
from mcp.client.stdio import stdio_client

async def main():
    async with stdio_client("python server.py") as (read, write):
        async with ClientSession(read, write) as session:

            # Discover tools
            tools = await session.list_tools()

            print("Available tools:")
            for tool in tools:
                print(tool.name)

asyncio.run(main())
```

This connects agent to MCP server.

---

# 5️⃣ Calling a Tool from MCP Client

Now let’s call tool.

```python
result = await session.call_tool(
    "get_weather",
    {"city": "Mumbai"}
)

print(result)
```

Server executes tool.

Returns result.

---

# 6️⃣ What Happens Internally

Behind the scenes:

Client sends:

```json
{
  "method": "tools/call",
  "params": {
    "name": "get_weather",
    "arguments": {
      "city": "Mumbai"
    }
  }
}
```

Server executes function.

Returns result.

Client forwards to agent.

---

# 7️⃣ Integrating with Your Agent (Important)

You already know agent structure:

```python
class Agent:
    tools = []
    llm = model
```

With MCP, tools become dynamic:

```python
tools = await session.list_tools()
```

Pass these schemas to LLM.

LLM decides tool.

Client executes tool.

---

# 8️⃣ Full Working Agent Example (Simplified)

```python
import asyncio
from mcp.client.session import ClientSession
from mcp.client.stdio import stdio_client

class MCPAgent:

    async def run(self):

        async with stdio_client("python server.py") as (read, write):

            async with ClientSession(read, write) as session:

                # discover tools
                tools = await session.list_tools()

                print("Discovered tools:", tools)

                # simulate LLM choosing tool
                result = await session.call_tool(
                    "get_weather",
                    {"city": "Delhi"}
                )

                print("Tool result:", result)

asyncio.run(MCPAgent().run())
```

This is minimal MCP-enabled agent.

---

# 9️⃣ How This Fits Into ReAct Cycle

Your ReAct loop becomes:

```text
User input
   ↓
LLM reasoning
   ↓
Select MCP tool
   ↓
MCP client executes tool
   ↓
Observation returned
   ↓
LLM continues reasoning
```

Exactly same ReAct pattern.

Only tool execution layer changes.

---

# 🔟 Resources Access Example

Reading resource:

```python
resource = await session.read_resource("customer://123")

print(resource)
```

No function call.

Direct resource access.

---

# 1️⃣1️⃣ Prompts Access Example

```python
prompt = await session.get_prompt(
    "review_code",
    {"code": "print('hello')"}
)
```

Returns reusable prompt template.

---

# 1️⃣2️⃣ Production Architecture Example

Real production agent:

```text
Agent
 ├── MCP Client
      ├── filesystem MCP server
      ├── database MCP server
      ├── github MCP server
      └── internal tools MCP server
```

Agent becomes extremely powerful.

---

# 1️⃣3️⃣ Key Insight (VERY IMPORTANT)

Your agent no longer owns tools.

It connects to tool ecosystem.

This makes agent:

* modular
* scalable
* extensible

---

# 1️⃣4️⃣ How ChatGPT / Claude / Cursor Work Internally

They connect to MCP servers like:

* filesystem
* git
* browser
* database

This is exactly how modern AI systems work.

You are now learning production-level architecture.

---

# Analogy — Installing Apps on Your Phone 📱

Your phone (Agent) does not contain all apps.

You install apps when needed.

Apps live outside core OS.

Same with MCP:

Agent = Phone
MCP Server = App Store
Tools = Apps
MCP Client = App installer

Agent installs and uses tools dynamically.

---

# Chapter 6 Summary

You learned:

* How MCP client connects to server
* How tools are discovered dynamically
* How tools are called
* How to integrate MCP with agent
* How ReAct cycle works with MCP



my understandings :

for one agent one mcp client , if multiple agents mcp agents multiple mcp clients

mcp server is like api gateway in microservices , but api gateway just routes to services and autherization and authenticatin

here mcp server gives you list of tools and executes the given tools


Excellent question. This is where most people get confused.

Short answer: **Yes, you *can* write Python code to talk directly to an MCP server. But the MCP client exists to handle all the protocol complexity, state, and integration with the agent brain automatically.**

Let’s break it down properly.

---

# First: You are correct — client is technically optional

You could write something like:

```python
# pseudo example
response = requests.post("http://mcp-server/tools/list")
tools = response.json()

response = requests.post("http://mcp-server/tools/call", json={
    "name": "get_balance",
    "arguments": {"account_id": "123"}
})
```

This works.

So why MCP Client?

Because real MCP is MUCH more complex than simple HTTP.

---

# The real reason MCP Client exists

The MCP client handles 5 major responsibilities:

---

## 1. Protocol handling (JSON-RPC, not simple REST)

MCP uses **JSON-RPC protocol**, not normal REST.

Example MCP message:

```json
{
  "jsonrpc": "2.0",
  "id": 1,
  "method": "tools/list"
}
```

Client must handle:

* request IDs
* matching responses
* async messages
* streaming
* errors

If you write manually → lots of boilerplate.

Client does this automatically.

---

## 2. Maintains persistent connection

MCP often uses:

* stdio
* WebSocket
* persistent connection

Not simple request-response.

Why?

Because server can send updates anytime.

Example:

```json
{
  "method": "tools/updated"
}
```

Without client, you must manage:

* connection lifecycle
* reconnect
* message parsing

Client handles this.

---

## 3. Converts tools into LLM-compatible format

This is VERY important.

MCP tool format:

```json
{
  "name": "get_balance",
  "inputSchema": {
    "type": "object",
    "properties": {
      "account_id": {"type": "string"}
    }
  }
}
```

LLM tool format (OpenAI style):

```json
{
  "type": "function",
  "function": {
    "name": "get_balance",
    "parameters": {...}
  }
}
```

Client converts MCP → LLM format automatically.

Without client, YOU must convert.

---

## 4. Executes tool calls on behalf of LLM

Flow:

```text
LLM says: call get_balance
Client detects tool call
Client sends request to MCP server
Client gets result
Client sends result back to LLM
```

Without client, YOU must write orchestration logic.

This becomes complex quickly.

---

## 5. Security and isolation

Client controls:

* which tools agent can access
* permission filtering
* sandboxing

Example:

Agent allowed tools:

```text
read_file
```

Blocked tools:

```text
delete_file
```

Client enforces this.

Without client → unsafe.

---

# The most important conceptual reason

The MCP client is the **bridge between LLM and MCP server**.

LLM cannot directly speak MCP protocol.

LLM only understands:

```json
tools = [...]
```

and

```json
tool_call = {...}
```

Client translates between:

```text
LLM world  ⇄  MCP world
```

---

# Visual explanation

Without MCP client:

```text
Agent Brain
   ↓
YOU must write protocol, conversion, execution logic
   ↓
MCP Server
```

With MCP client:

```text
Agent Brain
   ↓
MCP Client (handles everything)
   ↓
MCP Server
```

---

# Real analogy: Database driver

You *can* talk to PostgreSQL using raw TCP sockets.

But instead we use:

```python
psycopg2.connect()
```

Why?

Because driver handles:

* protocol
* parsing
* connection
* errors

MCP client = database driver
MCP server = database

---

# Another critical reason: LLM tool loop automation

Without client, you must manually implement this loop:

```python
while True:
    response = llm.ask(question, tools)

    if response.has_tool_call:
        result = call_mcp_server(response.tool_call)

        question = result
    else:
        break
```

Client does this automatically.

---

# When you WOULD write your own MCP client

You write custom client when building:

* custom AI agent framework
* custom IDE agent
* custom autonomous agent

Frameworks like:

* Claude Desktop → has MCP client
* Cursor → has MCP client
* LangChain → has MCP client support

---

# When you DON'T need to write MCP client

If using:

* Claude Desktop
* Cursor
* VSCode AI extensions

Client already exists.

You only write MCP server.

---

# Simple summary (very important)

You asked:

> why we need client?

Answer:

MCP client exists to handle:

| Responsibility        | Without Client | With Client |
| --------------------- | -------------- | ----------- |
| Protocol handling     | manual         | automatic   |
| Connection management | manual         | automatic   |
| Tool conversion       | manual         | automatic   |
| Tool execution loop   | manual         | automatic   |
| Security control      | manual         | automatic   |
| LLM integration       | hard           | easy        |

---

# Final mental model

MCP Server = Tool Provider
MCP Client = Tool Manager + Translator
LLM Brain = Decision Maker

---

# Most accurate one-line definition

**MCP Client is a protocol adapter that allows LLMs to use MCP servers safely and automatically.**

---

If you want, I can now show a **real Python MCP client and MCP server example (working code)** so you see exactly why client is needed.
